In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import random
from datasets import load_dataset

# Load Dataset

In [4]:
dataset = load_dataset("mrdbourke/FoodExtract-1k")

In [5]:
def get_random_idx(dataset):
    return random.randint(0, len(dataset['test']) - 1)

def get_random_sample(dataset):

    random_idx = get_random_idx(dataset)
    random_sample = dataset['test'][random_idx]

    example_input = random_sample['sequence']
    example_output = random_sample['gpt-oss-120b-label']
    example_output_condensed = random_sample['gpt-oss-120b-label-condensed']

    return random_sample

In [6]:
def sample_to_conversation_batch(batch):
    prompts = []
    completion = []

    for i in range(0, len(batch['sequence'])):
        # print(i)

        prompts.append(
                [{"role": "user", "content": batch["sequence"][i]}] # Load the sequence from the dataset
        )
         
        completion.append(
                [{"role": "assistant", "content": batch["gpt-oss-120b-label-condensed"][i]}] # Load the gpt-oss-120b generated label
        )

    return {
        "prompt": prompts,
        "completion": completion
    }

In [7]:
dataset = dataset.map(sample_to_conversation_batch,
                      batched=True)

In [8]:
dataset = dataset['train'].train_test_split(test_size=0.2,
                                       shuffle=False,
                                       seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 1136
    })
    test: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 284
    })
})

# Load model

In [9]:
MODEL_DIR = "best_model"

print(f"Loading model from {MODEL_DIR}...")

tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=MODEL_DIR
)


loaded_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_DIR,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

loaded_model

Loading model from best_model...


Loading weights: 100%|██████████| 236/236 [00:04<00:00, 52.44it/s] 


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [10]:
loaded_model_pipeline = pipeline(
    task="text-generation",
    model=loaded_model,
    tokenizer=tokenizer
)

loaded_model_pipeline

TextGenerationPipeline: {'model': 'Gemma3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': ('image', 'text'), 'output_modalities': ('text',)}

## Test Model on random sample

In [11]:
sample = get_random_sample(dataset)
sample

{'sequence': 'Nutrition Information Panel for Pizza Base:\n\n- Servings Per Package: 4 (half of one pizza base)\n- Serving Size: 100 g\n\nNutritional Details:\n- Energy: 1100 kJ / 263 Cal (13% Daily Intake)\n- Protein: 9.0 g (18% Daily Intake)\n- Fat, total: 3.0 g (4% Daily Intake)\n  - Saturated: 0.8 g (3% Daily Intake)\n- Carbohydrate: 48.0 g (15% Daily Intake)\n  - Sugars: 6.3 g (7% Daily Intake)\n- Dietary fibre: 2.2 g (7% Daily Intake)\n- Sodium: 524 mg (23% Daily Intake)\n\nIngredients: Wheat flour (60%), thiamine, folic acid, water, sugar, yeast, iodised salt, acidity regulator (297), preservative (282).\n\nContains: Wheat, gluten.\nMay contain: Milk.',
 'image_url': None,
 'class_label': 'food',
 'source': 'manual_taken_images',
 'char_len': 634.0,
 'word_count': 97.0,
 'syn_or_real': 'syn',
 'uuid': '9591ee17-40b9-498a-8aab-1e8d8817cebb',
 'gpt-oss-120b-label': "{'is_food_or_drink': True, 'tags': ['np', 'il', 'fi'], 'food_items': ['Pizza Base', 'Wheat flour', 'water', 'sugar',

In [12]:
sample_input = sample['gpt-oss-120b-label-condensed']
print(sample_input)

food_or_drink: 1
tags: np, il, fi
foods: Pizza Base, Wheat flour, water, sugar, yeast, iodised salt, milk
drinks:


In [13]:
sample = get_random_sample(dataset)

print(f"\n\nInput: {sample['prompt'][0]['content']}\n")

input_prompts = loaded_model_pipeline.tokenizer.apply_chat_template(
    conversation=sample["prompt"],
    tokenize=False,
    add_generation_prompt=True
)

default_outputs = loaded_model_pipeline(
    input_prompts,
    max_new_tokens=100
)

print(f"\n\nOutput: {sample['gpt-oss-120b-label-condensed']}")

print(f"\n\nPredicted Output: {default_outputs[0]['generated_text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.




Input: The image is a detailed screenshot of a Google Activity page displayed in the background. In the top left corner, there are three horizontal lines followed by the label "Google My Activity." Below this heading, on the left-hand side, there are several options listed in a grayed-out format: "Bundle View," "Item View," "Delete Activity by," "Other Google Activity," "Activity Controls," "Google Account," "Help and Feedback." The "Item View" option is currently highlighted.

Superimposed over this background is a bright white overlay screen featuring a list of options with checkboxes next to them. The first option's label is obscured, while the subsequent options are "Help," "Image Search," "Maps," "News," "Search," "Video Search," and "YouTube." The checkbox next to "YouTube" is ticked. At the bottom of this overlay, there are two blue buttons: "Cancel" and "Next."

On the right side of the image, the background screen shows a visible dropdown menu labeled "Delete." Additionally,

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




Output: food_or_drink: 0
tags: 
foods: 
drinks:


Predicted Output: <bos><start_of_turn>user
The image is a detailed screenshot of a Google Activity page displayed in the background. In the top left corner, there are three horizontal lines followed by the label "Google My Activity." Below this heading, on the left-hand side, there are several options listed in a grayed-out format: "Bundle View," "Item View," "Delete Activity by," "Other Google Activity," "Activity Controls," "Google Account," "Help and Feedback." The "Item View" option is currently highlighted.

Superimposed over this background is a bright white overlay screen featuring a list of options with checkboxes next to them. The first option's label is obscured, while the subsequent options are "Help," "Image Search," "Maps," "News," "Search," "Video Search," and "YouTube." The checkbox next to "YouTube" is ticked. At the bottom of this overlay, there are two blue buttons: "Cancel" and "Next."

On the right side of the imag

In [14]:
import time

def pred_on_text(input_text):
    start_time = time.time()
    
    raw_output = loaded_model_pipeline(text_inputs=[{"role": "user",
                                                    "content": input_text}],
                                       max_new_tokens=256,
                                       disable_compile=True)
    end_time = time.time()
    total_time = round(end_time - start_time, 4)

    generated_text = raw_output[0]["generated_text"][1]["content"]

    return generated_text, raw_output, total_time

example_input_text = "British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, a cup of tea and toast and fried tomatoes"
output_text, output_raw, output_total_time = pred_on_text(input_text=example_input_text)

print(f"[INFO] Input text:\n{example_input_text}\n")
print(f"[INFO] Model generated output text:\n{output_text}\n")
print(f"[INFO] Model full raw output:\n{output_raw}\n")
print(f"[INFO] Inference time:\n{output_total_time}\n")

[transformers] Passing `generation_config` together with generation-related arguments=({'disable_compile', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Input text:
British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, a cup of tea and toast and fried tomatoes

[INFO] Model generated output text:
food_or_drink: 1
tags: fi, di
foods: British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, toast and fried tomatoes
drinks: tea

[INFO] Model full raw output:
[{'generated_text': [{'role': 'user', 'content': 'British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, a cup of tea and toast and fried tomatoes'}, {'role': 'assistant', 'content': 'food_or_drink: 1\ntags: fi, di\nfoods: British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, toast and fried tomatoes\ndrinks: tea'}]}]

[INFO] Inference time:
3.0245



## Upload model on Hugging Face

In [18]:
from huggingface_hub import create_repo, upload_folder

HF_USERNAME = "bakarqureshi"

repo_id = f"{HF_USERNAME}/FoodExtract-gemma-3-270m-fine-tune-v1"

create_repo(repo_id=repo_id,
            repo_type="model",
            exist_ok=True, 
            private=False)

RepoUrl('https://huggingface.co/bakarqureshi/FoodExtract-gemma-3-270m-fine-tune-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='bakarqureshi/FoodExtract-gemma-3-270m-fine-tune-v1')

In [17]:
model_readme = r"""---
base_model: google/gemma-3-270m-it
library_name: transformers
model_name: checkpoint_models
tags:
- generated_from_trainer
- sft
- trl
license: gemma
---

# FoodExtract-v1

This is a food and drink extraction language model built on [Gemma 3 270M](https://huggingface.co/google/gemma-3-270m-it).

Given raw text, it's designed to:

1. Classify the text into food or drink (e.g. "a photo of a dog" = not food or drink, "a photo of a pizza" = food or drink).
2. Tag the text with one or more tags (see tags_dict below).
3. Extract the edible food-related items as a list.
4. Extract the edible drink-related items as a list.

For example, the input text might be:

```
British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, a cup of tea and toast and fried tomatoes
```

And the model will generate:

```
food_or_drink: 1
tags: fi, di
foods: British Breakfast, baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, toast, fried tomatoes
drinks: tea
```

This model can be used for filtering a large image caption (e.g. [DataComp-1B](https://huggingface.co/datasets/UCSC-VLAA/Recap-DataComp-1B)) text dataset for food and drink related items.

## Dataset

The model was trained on the [FoodExtract-1k](https://huggingface.co/datasets/mrdbourke/FoodExtract-1k) dataset.

This dataset contains 1400 samples of raw text and JSON output pairs of structured food extractions provided by `gpt-oss-120b`.

For example, a raw image caption input might be:

```
another optional quest takes place on windfall island during the night time play the song of passing a number of times and each time, glance towards the sky
```

And the `gpt-oss-120b` generated output (JSON) would be:

```
{'is_food_or_drink': 'false', 'tags': [], 'food_items': [], 'drink_items': []}
```

This is condensed to:

```
food_or_drink: 0\ntags: \nfoods: \ndrinks:
```

### Tags dictionary mapping

These tags are designed for fast filtering.

For example, the model can assign a certain tag based on what's in the raw text and then we can filter for "ingredient list" items.

```
tags_dict = {'np': 'nutrition_panel',
 'il': 'ingredient list',
 'me': 'menu',
 're': 'recipe',
 'fi': 'food_items',
 'di': 'drink_items',
 'fa': 'food_advertistment',
 'fp': 'food_packaging'}
```

## Helper functions

The model is trained to output a condensed version of the structured data.

We do this so the model can generate less tokens (e.g. it doesn't have to generate JSON outputs).

The following functions help to condense and uncondense raw text outputs/inputs into the desired structure.

```python
def condense_output(original_output):
    '''Helper function to condense a given FoodExtract string.
    
    Example input: {'is_food_or_drink': True, 'tags': ['fi'], 'food_items': ['cape gooseberries', 'mulberry', 'chilli powder', 'flathead lobster', 'hoisin sauce', 'duck leg', 'chestnuts', 'raw quail', 'duck breast', 'rogan josh curry sauce', 'brown rice', 'dango'], 'drink_items': []}

    Example output: food_or_drink: 1\ntags: fi\nfoods: cape gooseberries, mulberry, chilli powder, flathead lobster, hoisin sauce, duck leg, chestnuts, raw quail, duck breast, rogan josh curry sauce, brown rice, dango\ndrinks:'''

    condensed_output_string_base = '''food_or_drink: <is_food_or_drink>
    tags: <output_tags>
    foods: <food_items>
    drinks: <drink_items>'''

    is_food_or_drink = str(1) if str(original_output["is_food_or_drink"]).lower() == "true" else str(0)
    tags = ", ".join(original_output["tags"]) if len(original_output["tags"]) > 0 else ""
    foods = ", ".join(original_output["food_items"]) if len(original_output["food_items"]) > 0 else ""
    drinks = ", ".join(original_output["drink_items"]) if len(original_output["drink_items"]) > 0 else ""

    condensed_output_string_formatted = condensed_output_string_base.replace("<is_food_or_drink>", is_food_or_drink).replace("<output_tags>", tags).replace("<food_items>", foods).replace("<drink_items>", drinks)

    return condensed_output_string_formatted.strip()

def uncondense_output(condensed_output):
    '''Helper to go from condensed output to uncondensed output.

    Example input: food_or_drink: 1\ntags: fi\nfoods: cape gooseberries, mulberry, chilli powder, flathead lobster, hoisin sauce, duck leg, chestnuts, raw quail, duck breast, rogan josh curry sauce, brown rice, dango\ndrinks:

    Example output: {'is_food_or_drink': True, 'tags': ['fi'], 'food_items': ['cape gooseberries', 'mulberry', 'chilli powder', 'flathead lobster', 'hoisin sauce', 'duck leg', 'chestnuts', 'raw quail', 'duck breast', 'rogan josh curry sauce', 'brown rice', 'dango'], 'drink_items': []}
    '''

    condensed_list = condensed_output.split("\n")

    condensed_dict_base = {
        "is_food_or_drink": "",
        "tags": [],
        "food_items": [],
        "drink_items": []
    }

    # Set values to defaults
    food_or_drink_item = None
    tags_item = None
    foods_item = None
    drinks_item = None

    # Extract items from condensed_list
    for item in condensed_list:
        if "food_or_drink:" in item.strip():
            food_or_drink_item = item

        if "tags:" in item:
            tags_item = item

        if "foods:" in item:
            foods_item = item

        if "drinks:" in item:
            drinks_item = item

    if food_or_drink_item:
        is_food_or_drink_bool = True if food_or_drink_item.replace("food_or_drink: ", "").strip() == "1" else False
    else:
        is_food_or_drink_bool = None

    if tags_item:
        tags_list = [item.replace("tags: ", "").replace("tags:", "").strip() for item in tags_item.split(", ")]
        tags_list = [item for item in tags_list if item] # Filter for empty items
    else:
        tags_list = []

    if foods_item:
        foods_list = [item.replace("foods:", "").replace("foods: ", "").strip() for item in foods_item.split(", ")]
        foods_list = [item for item in foods_list if item] # Filter for empty items
    else:
        foods_list = []

    if drinks_item:
        drinks_list = [item.replace("drinks:", "").replace("drinks: ", "").strip() for item in drinks_item.split(", ")]
        drinks_list = [item for item in drinks_list if item] # Filter for empty items
    else:
        drinks_list = []

    condensed_dict_base["is_food_or_drink"] = is_food_or_drink_bool
    condensed_dict_base["tags"] = tags_list
    condensed_dict_base["food_items"] = foods_list
    condensed_dict_base["drink_items"] = drinks_list

    return condensed_dict_base
````

## Quick start

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_PATH = "mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1"

# Load the model into a pipeline
loaded_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
)

# Create model pipeline
loaded_model_pipeline = pipeline("text-generation",
                                 model=loaded_model,
                                 tokenizer=tokenizer)

# Create a sample to predict on
input_text = "A plate with bacon, eggs and toast on it"
input_text_user = [{'content': input_text, 'role': 'user'}]

# Apply the chat template
input_prompt = loaded_model_pipeline.tokenizer.apply_chat_template(conversation=input_text_user,
                                                                    tokenize=False,
                                                                    add_generation_prompt=True)

# Let's run the default model on our input
default_outputs = loaded_model_pipeline(text_inputs=input_prompt, 
                                        max_new_tokens=256)

# View the outputs
print(f"[INFO] Test sample input:\n{input_prompt}\n")
print(f"[INFO] Fine-tuned model output:\n{default_outputs[0]['generated_text'][len(input_prompt):]}\n")
```

You should see an output similar to:

```
[INFO] Test sample input:
<bos><start_of_turn>user
A plate with bacon, eggs and toast on it<end_of_turn>
<start_of_turn>model

[INFO] Fine-tuned model output:
food_or_drink: 1
tags: fi
foods: bacon, eggs, toast
drinks:
```

## Training procedure

This model was trained with SFT (Supervised Fine-Tuning) via Hugging Face's TRL library.

See the full training walkthrough at: https://www.learnhuggingface.com/notebooks/hugging_face_llm_full_fine_tune_tutorial

## Citations

* Reference for structured data extraction was taken from the paper [*Essential-Web v1.0: 24T tokens of organized web data*](https://huggingface.co/papers/2506.14111) and the subsequent fine-tuned model [`EssentialAI/eai-distill-0.5b`](https://huggingface.co/EssentialAI/eai-distill-0.5b)."""

# Save README to file
with open(f"{MODEL_DIR}/README.md", "w") as f:
    f.write(model_readme)

In [36]:
# command = f"hf upload {MODEL_DIR} --repo_id {repo_id} --repo_type model --token ${{HF_TOKEN}}"
HF_HUB_DISABLE_XT=1
command = f"hf upload {repo_id} {MODEL_DIR}"
!{command}

Found 8 files to upload
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
Traceback (most recent call last):
  File "/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/bin/hf", line 6, in <module>
    sys.exit(main())
  File "/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/huggingface_hub/cli/hf.py", line 133, in main
    app()
  File "/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/click/core.py", line 1569, in __call__
    return self.main(*args, **kwargs)
  File "/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/click/core.py", line 1490, in main
    rv = self.invoke(ctx)
  File "/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/huggingface_hub/cli/_cli_utils.py", line 131, in invo

In [37]:
print(f"[INFO] Uploading model directory: {MODEL_DIR} to Hugging Face repo: {repo_id}")

upload_folder(
    folder_path=MODEL_DIR,
    repo_id=repo_id,
    repo_type="model"
)

[INFO] Uploading model directory: best_model to Hugging Face repo: bakarqureshi/FoodExtract-gemma-3-270m-fine-tune-v1


ConnectionError: Network error: Request error: HTTP status client error (403 Forbidden), domain: https://huggingface.co/api/models/bakarqureshi/FoodExtract-gemma-3-270m-fine-tune-v1/xet-write-token/main

In [1]:
%%writefile demos/FoodExtract/app.py

# Load dependencies
import time
import transformers
import torch
import spaces # Optional: run our model on the GPU (this will be much faster inference)

import gradio as gr

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline

@spaces.GPU # Optional: run our model on the GPU (this will be much faster inference)
def pred_on_text(input_text):
    start_time = time.time()
    
    raw_output = loaded_model_pipeline(text_inputs=[{"role": "user",
                                                    "content": input_text}],
                                       max_new_tokens=256,
                                       disable_compile=True)
    end_time = time.time()
    total_time = round(end_time - start_time, 4)

    generated_text = raw_output[0]["generated_text"][1]["content"]

    return generated_text, raw_output, total_time

# Load the model (from our Hugging Face Repo)
# Note: You may have to replace my username `mrdbourke` for your own
MODEL_PATH = "mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1"

# Load the model into a pipeline
loaded_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
)

# Create model pipeline
loaded_model_pipeline = pipeline("text-generation",
                                 model=loaded_model,
                                 tokenizer=tokenizer)

# Create the demo
description = """Extract food and drink items from text with a fine-tuned SLM (Small Language Model) or more specifically a fine-tuned [Gemma 3 270M](https://huggingface.co/google/gemma-3-270m-it).

Our model has been fine-tuned on the [FoodExtract-1k dataset](https://huggingface.co/datasets/mrdbourke/FoodExtract-1k). 

* Input (str): Raw text strings or image captions (e.g. "A photo of a dog sitting on a beach" or "A breakfast plate with bacon, eggs and toast")
* Output (str): Generated text with food/not_food classification as well as noun extracted food and drink items and various food tags.

For example:

* Input: "For breakfast I had eggs, bacon and toast and a glass of orange juice"
* Output: 

```
food_or_drink: 1
tags: fi, di
foods: eggs, bacon, toast
drinks: orange juice
```

See full fine-tuning code at [learnhuggingface.com](https://www.learnhuggingface.com/notebooks/hugging_face_llm_full_fine_tune_tutorial).
"""

# Create the Gradio text in and out interface
demo = gr.Interface(fn=pred_on_text,
                    inputs=gr.TextArea(lines=4, label="Input Text"),
                    outputs=[gr.TextArea(lines=4, label="Generated Text"),
                             gr.TextArea(lines=7, label="Raw Output"),
                             gr.Number(label="Generation Time (s)")],
                    title="🍳 Structured FoodExtract with a Fine-Tuned Gemma 3 270M",
                    description=description,
                    examples=[["Hello world! This is my first fine-tuned LLM!"],
                              ["A plate of food with grilled barramundi, salad with avocado, olives, tomatoes and Italian dressing"],
                              ["British Breakfast with baked beans, fried eggs, black pudding, sausages, bacon, mushrooms, a cup of tea and toast and fried tomatoes"],
                              ["Steak tacos"],
                              ["A photo of a dog sitting on a beach"]]
)

if __name__ == "__main__":
    demo.launch(share=False)

Writing demos/FoodExtract/app.py


In [2]:
%%writefile demos/FoodExtract/README.md
---
title: FoodExtract Fine-tuned LLM Structued Data Extractor v1
emoji: 📝➡️🍟
colorFrom: green
colorTo: blue
sdk: gradio
app_file: app.py
pinned: false
license: apache-2.0
---

"""
Fine-tuned Gemma 3 270M to extract food and drink items from raw text.

Input can be any form of real text (mostly focused on shorter image caption-like texts):

```
A truly eclectic and mouth-watering feast is laid out on the table, featuring savory favorites like crispy fried chicken, 
a perfectly seared steak, and loaded tacos, complete with a side of creamy mayonnaise. To balance the heavier mains, 
a vibrant assortment of fresh fruit sits nearby, including a crisp red apple, a tropical pineapple, and a scattering of 
sweet cherries. Thirst-quenching options complete this extravagant spread, with a classic iced latte, an earthy matcha latte, 
and a simple, refreshing glass of milk ready to be enjoyed.
```

And output will be a formatted string such as the following:

```
food_or_drink: 1
tags: fi, re
foods: tacos,red apple, pineapple, cherries, fried chicken, steak, mayonnaise
drinks: iced latte, matcha latte, milk
```

The tags map to the following items:

```
tags_dict = {'np': 'nutrition_panel',
 'il': 'ingredient list',
 'me': 'menu',
 're': 'recipe',
 'fi': 'food_items',
 'di': 'drink_items',
 'fa': 'food_advertistment',
 'fp': 'food_packaging'}
```

* You can see walkthrough step by step code details at: https://www.learnhuggingface.com/notebooks/hugging_face_llm_full_fine_tune_tutorial 
* See the fine-tuning dataset: https://huggingface.co/datasets/mrdbourke/FoodExtract-1k
* See the fine-tuned model: https://huggingface.co/mrdbourke/FoodExtract-gemma-3-270m-fine-tune-v1
"""

Writing demos/FoodExtract/README.md


In [3]:
%%writefile demos/FoodExtract/requirements.txt
transformers
gradio
torch
accelerate

Writing demos/FoodExtract/requirements.txt
